# Butina Clustering Acquisition Validation

This notebook validates the Butina clustering acquisition method for diverse molecular selection.
Butina clustering creates non-hierarchical clusters based on molecular similarity thresholds.


In [ ]:
# Cell 1: Imports & Configuration
import sys
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
import logging
from datetime import datetime

# Jupyter notebook configuration
%matplotlib inline
%load_ext autoreload
%autoreload 2

# Suppress warnings for cleaner output
warnings.filterwarnings('ignore')

# Add project root to path
PROJECT_ROOT = Path().resolve().parent.parent
sys.path.append(str(PROJECT_ROOT))

from validation.create_embedding_plots import create_embedding_plots
from validation.data_loading import setup_data_with_error_handling

# LearnM8 imports
from learnm8.oracles import CSVOracle
from learnm8.core.data_manager import DataManager
from learnm8.learners.ensemble import RFEnsemble
from learnm8.utils.data_loaders import load_benchmark_data

# Try to import Butina clustering acquisition
try:
    from learnm8.acquisition import ButinaClusteringAcquisition
    BUTINA_AVAILABLE = True
    print("✅ Butina clustering acquisition available")
except ImportError as e:
    BUTINA_AVAILABLE = False
    print(f"❌ Butina clustering not available: {e}")
    print("Make sure RDKit is installed: conda install -c conda-forge rdkit")

In [ ]:
# Cell 2: Load Dataset
print("📊 Step 2: Loading dataset...")
target = "ADA"
target_path = "/home/tony/LearnM8/ESSENCE_benchmark_input/ADA.csv"
target_column = "ESSENCE-Dock_Score"

try:
    compound_pool, ground_truth = load_benchmark_data(str(target_path), target_column)
    print(f"Loaded {len(compound_pool)} compounds")
    print(f"Target column: {target_column}")
    print(f"Score range: {ground_truth[target_column].min():.2f} to {ground_truth[target_column].max():.2f}")
    
except Exception as e:
    print(f"Failed to load dataset: {e}")
    print(f"\n❌ Error loading dataset: {e}")

In [ ]:
# Cell 3: DataManager Setup
print("⚙️  Step 3: Setting up data manager with error handling...")

# Initialize DataManager
data_manager = DataManager(results_dir=str(Path("./")), featurizer='morgan')

# Test and setup data with error handling
compound_pool = setup_data_with_error_handling(compound_pool, data_manager)

In [ ]:
# Cell 4: Initial Training
print("🎯 Step 4: Performing initial training...")

learner = RFEnsemble(n_estimators=3, random_states=[42, 43, 44])

# Determine initial training size
initial_size = min(100, len(compound_pool) // 10)  # Adaptive initial size
np.random.seed(42)
initial_indices = np.random.choice(len(compound_pool), size=initial_size, replace=False)

# Split data
labeled_compounds = compound_pool.iloc[initial_indices].copy()
unlabeled_compounds = compound_pool.drop(compound_pool.index[initial_indices]).copy()

# Create oracle and measure labeled compounds
oracle = CSVOracle(csv_path=str(target_path))
labeled_compounds = oracle.measure(labeled_compounds, [target_column])

# Train the model
learner.train(labeled_compounds, target_column, data_manager)
print(f"Initial training completed: {len(labeled_compounds)} labeled, {len(unlabeled_compounds)} unlabeled")

In [ ]:
# Cell 5: Butina Clustering Acquisition
if not BUTINA_AVAILABLE:
    print("❌ Butina clustering not available - skipping acquisition")
else:
    print("🎯 Step 5: Running Butina clustering acquisition...")
    
    # Set acquisition parameters
    batch_fraction = 0.01
    batch_size = max(1, int(batch_fraction * len(unlabeled_compounds)))
    
    # Limit dataset size for Butina clustering (O(n²) complexity)
    max_butina_size = 1000
    if len(unlabeled_compounds) > max_butina_size:
        print(f"⚠️  Dataset too large ({len(unlabeled_compounds)}). Using random subset of {max_butina_size} compounds for Butina clustering.")
        np.random.seed(42)
        subset_indices = np.random.choice(len(unlabeled_compounds), size=max_butina_size, replace=False)
        unlabeled_subset = unlabeled_compounds.iloc[subset_indices].copy()
    else:
        unlabeled_subset = unlabeled_compounds.copy()
    
    print(f"Batch size: {batch_size}")
    print(f"\n🎯 Acquisition Parameters:")
    print(f"  • Batch fraction: {batch_fraction*100:.1f}%")
    print(f"  • Batch size: {batch_size:,}")
    print(f"  • Unlabeled pool: {len(unlabeled_subset):,}")
    print(f"  • Similarity threshold: 0.4 (60% similarity cutoff)")
    
    # Get predictions for unlabeled compounds
    print("Getting predictions for unlabeled compounds...")
    predictions, uncertainties = learner.predict(unlabeled_subset, data_manager)
    
    # Prepare unlabeled compounds with predictions
    unlabeled_with_predictions = unlabeled_subset.copy()
    unlabeled_with_predictions['prediction'] = predictions
    if uncertainties is not None:
        unlabeled_with_predictions['uncertainty'] = uncertainties
    else:
        unlabeled_with_predictions['uncertainty'] = np.zeros(len(predictions))
    
    # Initialize Butina clustering acquisition
    print("Initializing Butina clustering acquisition function...")
    init_start_time = datetime.now()
    butina_acquisition = ButinaClusteringAcquisition(
        threshold=0.4,  # 60% similarity cutoff
        featurizer_type='morgan',
        fp_radius=2,
        fp_size=1024,
        random_state=42
    )
    init_time = datetime.now() - init_start_time
    print(f"Initialization time: {init_time.total_seconds():.2f} seconds")
    
    # Perform acquisition
    print("Performing Butina clustering compound selection...")
    selection_start_time = datetime.now()
    try:
        selected_compounds = butina_acquisition.select(
            compounds=unlabeled_with_predictions, 
            n_select=batch_size
        )
        selected_indices = selected_compounds.index.values
        selection_time = datetime.now() - selection_start_time
        total_butina_time = init_time + selection_time
        
        print(f"Selection time: {selection_time.total_seconds():.2f} seconds")
        print(f"Butina clustering selected {len(selected_indices)} compounds")
        print(f"Cluster sizes (acquisition scores): {list(selected_compounds['acquisition_score'])}")
        
        SELECTION_SUCCESS = True
        
    except Exception as e:
        print(f"❌ Butina clustering failed: {e}")
        SELECTION_SUCCESS = False
        total_butina_time = init_time

In [ ]:
# Cell 6: Generate Embeddings & Visualizations
if BUTINA_AVAILABLE and SELECTION_SUCCESS:
    print("📊 Step 6: Generating visualizations...")
    
    plots_dir = Path(f"./plots_{target}")
    plots_dir.mkdir(exist_ok=True)
    
    # Generate PCA embeddings from molecular fingerprints
    print("Generating PCA embeddings from molecular fingerprints")
    features = data_manager.get_features(
        compound_ids=unlabeled_subset['ID'].tolist(),
        smiles_list=unlabeled_subset['SMILES'].tolist(),
        featurizer_type='morgan'
    )
    
    # Generate PCA embeddings
    pca = PCA(n_components=2, random_state=42)
    features_array = features.toarray() if hasattr(features, 'toarray') else features
    embeddings = pca.fit_transform(features_array)
    
    # For Butina clustering, we create pseudo-clusters based on acquisition scores
    # This is for visualization purposes since Butina doesn't produce 2D embeddings directly
    labels = np.zeros(len(unlabeled_subset))
    
    # Assign cluster labels to selected compounds based on their acquisition scores
    if len(selected_compounds) > 0:
        for i, (idx, score) in enumerate(zip(selected_indices, selected_compounds['acquisition_score'])):
            if idx in unlabeled_subset.index:
                pos = unlabeled_subset.index.get_loc(idx)
                labels[pos] = int(score)  # Use cluster size as pseudo-cluster ID
    
    # Convert selected indices to unlabeled dataset positions
    unlabeled_positions = {idx: pos for pos, idx in enumerate(unlabeled_subset.index)}
    selected_positions = [unlabeled_positions[idx] for idx in selected_indices if idx in unlabeled_positions]
    
    # Create visualization
    method_name = f"butina - {total_butina_time.total_seconds():.2f} seconds"
    create_embedding_plots(embeddings, labels, selected_positions, method_name, plots_dir)
    
    print(f"✅ Visualization saved to {plots_dir}/{method_name.replace(' ', '_').replace('-', '').lower()}_analysis.png")
    
else:
    print("⚠️  Skipping visualization due to failed acquisition or missing dependencies")

In [ ]:
# Cell 7: Cleanup
import shutil
# Clean up temporary files
try:
    shutil.rmtree(Path(data_manager.results_dir) / ".cache/", ignore_errors=True)
    print("🧹 Cleaned up temporary files")
except:
    pass

print("\n✅ Butina clustering validation completed!")
if BUTINA_AVAILABLE and SELECTION_SUCCESS:
    print(f"📊 Results: Selected {len(selected_compounds)} diverse compounds using Butina clustering")
    print(f"⏱️  Total time: {total_butina_time.total_seconds():.2f} seconds")
    print(f"🎯 Cluster diversity: {len(set(selected_compounds['acquisition_score']))} unique cluster sizes")